# Queue B: unconstrained and IterGen, both models

**Runtime: G4 GPU** (NVIDIA RTX PRO 6000; Runtime > Change runtime type), then
**Runtime > Run all**. Nothing to edit.

Runs unconstrained, IterGen with semantic checks off, and IterGen with checks on, on all 397 tasks, Qwen2.5 then Qwen3.5; each is evaluated with node when it finishes.

After a disconnect, Run all again: runs that finished are skipped, the unfinished one
continues where it stopped. Outputs go to the same Drive folders as
`run_final.ipynb` (`constrained-llm-js-codegen/final/<model>/<run>`), plus a
`queue_done.txt` in each finished run's folder.

When the queue ends, the notebook writes everything out to Drive and releases the
runtime by itself (`AUTO_DISCONNECT`), so it can run unattended; stopping it by hand
does not release it.

In [ ]:
QUEUE = [
    ('Qwen/Qwen2.5-3B-Instruct', 'unconstrained'),
    ('Qwen/Qwen2.5-3B-Instruct', 'itergen_off'),
    ('Qwen/Qwen2.5-3B-Instruct', 'itergen_on'),
    ('Qwen/Qwen3.5-4B', 'unconstrained'),
    ('Qwen/Qwen3.5-4B', 'itergen_off'),
    ('Qwen/Qwen3.5-4B', 'itergen_on'),
]
N_TASKS    = None                         # all 397
SAMPLE     = None
SEED       = 0
N_WORKERS  = 'auto'                       # 1, or for chopchop as many as fit
CHECK_CPU_LIMIT = 600                     # chopchop: per-task check CPU seconds (from the pilot)
WALL_CAP   = 1800                         # chopchop: per-task wall-clock cap, a safety net
EXPECT_GPU = 'RTX PRO 6000'               # G4; every final run uses the same GPU type
REPO_URL    = 'https://github.com/BowenX307/constrained-llm-js-codegen-final.git'
REPO_COMMIT = '6618ca975c0cbeaa7075976b72f1d88a28f718c6'
RUN_TAG     = '2c14eef'                   # output folder suffix
AUTO_DISCONNECT = True                    # release the runtime when the queue ends (saves compute units)

In [ ]:
import os, sys, json, time, shutil, glob, subprocess, threading
import psutil

REPO   = '/content/final'
CACHE  = '/content/cache'
WORK   = '/content/work'
DRIVE_ROOT = '/content/drive/MyDrive/constrained-llm-js-codegen'     # same folder as the earlier runs
MS_DRIVE   = f'{DRIVE_ROOT}/mask_store_zst'                          # compressed mask stores, reused
os.makedirs(CACHE, exist_ok=True)
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                     capture_output=True, text=True)
if smi.returncode != 0 or not smi.stdout.strip():
    raise RuntimeError('No GPU. Runtime > Change runtime type > G4 GPU, then Run all.')
GPU_NAME, _mb = [x.strip() for x in smi.stdout.strip().splitlines()[0].split(',')]
GPU_GB, RAM_GB, CPUS = int(float(_mb)) / 1024, psutil.virtual_memory().total / 2**30, os.cpu_count()
print(f'GPU {GPU_NAME} ({GPU_GB:.0f} GB) | RAM {RAM_GB:.0f} GB | {CPUS} CPUs')
if EXPECT_GPU not in GPU_NAME:
    raise RuntimeError(f'This runtime has {GPU_NAME}; every final run uses {EXPECT_GPU}. '
                       'Change the runtime type (or EXPECT_GPU, if that is really intended).')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive not mounted:', type(e).__name__)
DRIVE_OK = os.path.isdir('/content/drive/MyDrive')
if not DRIVE_OK:
    print('WARNING: no Drive -- a disconnect that recycles the VM loses the outputs.')

if os.path.isdir(REPO):
    shutil.rmtree(REPO)
for cmd in (['git', 'init', '-q', REPO], ['git', '-C', REPO, 'remote', 'add', 'origin', REPO_URL],
            ['git', '-C', REPO, 'fetch', '-q', '--depth', '1', 'origin', REPO_COMMIT],
            ['git', '-C', REPO, 'checkout', '-q', 'FETCH_HEAD']):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(' '.join(cmd) + ' failed:\n' + r.stderr)
print('code at', REPO_COMMIT[:7])

def pip(*a):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('pip install failed: ' + ' '.join(a) + '\n' + r.stderr[-2000:])
def importable(stmt):
    return subprocess.run([sys.executable, '-c', stmt], cwd=REPO, capture_output=True).returncode == 0

from importlib.metadata import version
if version('transformers') != '5.20.1':
    was = version('transformers')
    pip('transformers==5.20.1')
    if 'transformers' in sys.modules:    # the old version is already loaded in this session
        raise RuntimeError(f'transformers was {was}; installed 5.20.1. Runtime > Restart session, then Run all.')
import transformers
pip('lark', 'regex', 'networkx', 'fire', 'interegular', 'tqdm', 'datasets', 'jsonschema', 'psutil')
if not importable('from syncode.parsers.itergen_parser import IGParser'):
    # GitHub, pinned, --no-deps (so it cannot replace torch / transformers)
    pip('--no-deps', '--ignore-requires-python', '--force-reinstall',
        'git+https://github.com/structuredllm/syncode.git@4d6c110a028d3fb9cbb5f1a6ac6446b696f3eac4')
for tool, pkg in (('zstd', 'zstd'), ('node', 'nodejs')):
    if not shutil.which(tool):
        subprocess.run(['apt-get', 'install', '-y', '-qq', pkg], capture_output=True)
import torch
print('transformers', transformers.__version__, '| torch', torch.__version__,
      '| node', subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip())
ENV = dict(os.environ, SYNCODE_CACHE=CACHE + '/syncode/', HF_HOME=CACHE + '/hf_home',
           TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1', PYTHONPATH=REPO,
           HF_HUB_DISABLE_PROGRESS_BARS='1')

In [ ]:
import traceback
interrupted = False
try:
    for MODEL, MODE in QUEUE:
        print('=' * 100 + f'\n{MODEL}  {MODE}\n' + '=' * 100, flush=True)
        assert MODE in ('unconstrained', 'itergen_off', 'itergen_on', 'chopchop'), MODE
        SLUG   = MODEL.split('/')[-1]
        RUN    = (MODE + (f'_lim{CHECK_CPU_LIMIT:g}' if MODE == 'chopchop' else '')
                  + (f'_sample{SAMPLE}seed{SEED}' if SAMPLE else '') + '_' + RUN_TAG)
        OUT    = f'{WORK}/{SLUG}/{RUN}'
        DRIVE_OUT  = f'{DRIVE_ROOT}/final/{SLUG}/{RUN}'
        os.makedirs(OUT, exist_ok=True)
        if DRIVE_OK:
            os.makedirs(DRIVE_OUT, exist_ok=True)
            restored = 0
            for f in os.listdir(DRIVE_OUT):
                if f.endswith('.json') and not os.path.exists(f'{OUT}/{f}'):
                    shutil.copy(f'{DRIVE_OUT}/{f}', f'{OUT}/{f}'); restored += 1
            print(f'Drive: {restored} files restored from {DRIVE_OUT}')
        if DRIVE_OK and os.path.exists(f'{DRIVE_OUT}/queue_done.txt'):
            print('finished earlier: skipped'); continue
        try:
            if MODE.startswith('itergen'):
                GRAMMAR = f'{REPO}/generators/grammars/javascript.lark'
                probe = subprocess.run([sys.executable, '-c',
                    'import syncode.common as c; from syncode.parsers.grammars import Grammar\n'
                    f'tok = c.load_tokenizer({MODEL!r}); g = Grammar({GRAMMAR!r})\n'
                    'print("MS_PATH=" + c.SYNCODE_CACHE + "mask_stores/" + type(tok).__name__ + '
                    '"/grammar_strict_" + g.hash() + "_" + str(tok.vocab_size) + ".pkl")'],
                    env=ENV, cwd=REPO, capture_output=True, text=True)
                lines = [l for l in probe.stdout.splitlines() if l.startswith('MS_PATH=')]
                if probe.returncode != 0 or not lines:
                    raise RuntimeError('could not resolve the mask store path:\n' + probe.stderr[-2000:])
                MS_PATH = lines[-1][len('MS_PATH='):]
                MS_ZST = f'{MS_DRIVE}/{os.path.basename(MS_PATH)}.zst'
                os.makedirs(os.path.dirname(MS_PATH), exist_ok=True)
                print('mask store:', os.path.basename(MS_PATH))
                t0 = time.time()
                if not (os.path.exists(MS_PATH) and os.path.getsize(MS_PATH) > 100 * 2**20):
                    if DRIVE_OK and os.path.exists(MS_ZST):
                        r = subprocess.run(['zstd', '-q', '-d', '-f', '-T0', MS_ZST, '-o', MS_PATH + '.part'],
                                           capture_output=True, text=True)
                        if r.returncode != 0:
                            raise RuntimeError('the copy on Drive is unreadable: ' + r.stderr[-300:])
                        os.replace(MS_PATH + '.part', MS_PATH)
                        print(f'restored from Drive in {time.time() - t0:.0f}s')
                    else:
                        print('not on Drive: building it (~30 min) ...')
                        r = subprocess.run([sys.executable, '-c',
                            'import syncode.common as c\nfrom syncode.parsers.grammars import Grammar\n'
                            'from syncode.mask_store.mask_store import MaskStore\n'
                            f'tok = c.load_tokenizer({MODEL!r})\n'
                            f"MaskStore.init_mask_store(grammar=Grammar({GRAMMAR!r}), tokenizer=tok, use_cache=True, mode='grammar_strict')\n"],
                            env=ENV, cwd=REPO, capture_output=True, text=True)
                        if r.returncode != 0 or not os.path.exists(MS_PATH):
                            raise RuntimeError(f'build failed (exit {r.returncode}; -9 = out of RAM):\n' + r.stderr[-2000:])
                        print(f'built in {(time.time() - t0) / 60:.0f} min')
                        if DRIVE_OK:
                            os.makedirs(MS_DRIVE, exist_ok=True)
                            z = subprocess.run(['zstd', '-q', '-f', '-T0', '-3', MS_PATH, '-o', MS_ZST + '.part'],
                                               capture_output=True, text=True)
                            if z.returncode == 0:
                                os.replace(MS_ZST + '.part', MS_ZST)
                                print(f'saved to Drive: {os.path.getsize(MS_ZST) / 2**30:.2f} GB')
                            else:
                                print('*** could not save it to Drive (full?):', z.stderr[-200:])
                print(f'mask store ready: {os.path.getsize(MS_PATH) / 2**30:.2f} GB')
            else:
                print(MODE, 'does not use a mask store')
            import selectors

            if N_WORKERS != 'auto':
                NW = int(N_WORKERS)
            elif MODE == 'chopchop':
                # each worker holds its own copy of the model (~7-9 GB of GPU memory) and needs ~1 CPU
                NW = max(1, min(int(GPU_GB // 10), CPUS - 1, int(RAM_GB // 8)))
            else:
                NW = 1
            THREADS = max(1, CPUS // NW)
            CAP = WALL_CAP if MODE == 'chopchop' else 0
            STALL_S = CAP + 1800 if CAP else 3 * 3600 if MODE == 'chopchop' else 1800   # a worker silent this long is stuck

            def n_done():
                return len([f for f in os.listdir(OUT) if f.startswith('mbpp_') and f.endswith('.json')])
            N_TOTAL = min(x for x in (SAMPLE, N_TASKS, 397) if x)

            # the tasks not done yet, dealt out evenly to the workers (each gets its own list, kept on restart)
            SEL = (['--n_tasks', str(N_TASKS)] if N_TASKS else []) + (['--sample', str(SAMPLE), '--seed', str(SEED)] if SAMPLE else [])
            r = subprocess.run([sys.executable, f'{REPO}/run.py', '--model', MODEL, '--mode', MODE, '--out', OUT,
                                '--list_todo'] + SEL, capture_output=True, text=True, env=ENV, cwd=REPO)
            if r.returncode != 0:
                raise RuntimeError('run.py --list_todo failed:\n' + r.stderr[-2000:])
            TODO = r.stdout.split()
            NW = max(1, min(NW, len(TODO)))
            TASK_FILES = []
            for i in range(NW):
                TASK_FILES.append(f'{WORK}/todo_{SLUG}_{MODE}_{i}.txt')
                open(TASK_FILES[i], 'w').write('\n'.join(TODO[i::NW]) + '\n')
            print(f'{MODE}: {len(TODO)} tasks to do, {NW} worker(s), {THREADS} CPU threads each'
                  + (f', wall cap {CAP:g} s per task' if CAP else ''))

            stop_sync = threading.Event()
            def copy_to_drive():
                if not DRIVE_OK:
                    return
                for f in os.listdir(OUT):
                    if f.endswith('.json') or f.endswith('.jsonl'):
                        src, dst = f'{OUT}/{f}', f'{DRIVE_OUT}/{f}'
                        try:
                            if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
                                shutil.copyfile(src, dst)
                        except Exception as e:
                            print('Drive copy failed:', type(e).__name__, e)
            def sync_to_drive():
                while not stop_sync.wait(60):
                    copy_to_drive()

            def launch(i):
                cmd = [sys.executable, '-u', f'{REPO}/run.py', '--model', MODEL, '--mode', MODE, '--out', OUT,
                       '--tasks_file', TASK_FILES[i], '--threads', str(THREADS), '--check_cpu_limit', str(CHECK_CPU_LIMIT)] + SEL
                if CAP:
                    cmd += ['--wall_cap', str(CAP)]
                p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                                     env=ENV, cwd=REPO)
                sel.register(p.stdout, selectors.EVENT_READ, i)
                return p

            sync = threading.Thread(target=sync_to_drive, daemon=True); sync.start()
            sel = selectors.DefaultSelector()
            procs, last, restarts, fatal = {}, {}, {i: 0 for i in range(NW)}, None
            t0, done0 = time.time(), n_done()
            for i in range(NW):
                procs[i] = launch(i); last[i] = time.time()
            try:
                while procs:
                    for key, _ in sel.select(timeout=10):
                        i, line = key.data, key.fileobj.readline()
                        if not line:
                            sel.unregister(key.fileobj); continue
                        last[i] = time.time()
                        if line.startswith('[') and '/' in line.split(']')[0]:
                            d = n_done(); rate = (time.time() - t0) / max(1, d - done0)
                            print(f'w{i} {line.rstrip()}   | {d}/{N_TOTAL}, ETA {rate * (N_TOTAL - d) / 60:.0f} min', flush=True)
                        elif not any(s in line for s in ('Loading weights', 'Warning', 'warn(', 'torch_dtype')):
                            print(f'w{i}: {line.rstrip()}', flush=True)
                    for i, p in list(procs.items()):
                        if p.poll() is None and time.time() - last[i] > STALL_S:
                            print(f'!! worker {i} silent for {STALL_S} s: killing it'); p.kill(); p.wait()
                        rc = p.poll()
                        if rc is None:
                            continue
                        del procs[i]
                        if rc == 0:
                            continue
                        restarts[i] += 1
                        print(f'!! worker {i} exited with {rc} (restart {restarts[i]})', flush=True)
                        if restarts[i] > 5:
                            fatal = f'worker {i} failed more than 5 times'; continue
                        procs[i] = launch(i); last[i] = time.time()
            finally:
                for p in procs.values():
                    p.terminate()
                stop_sync.set(); sync.join()
                copy_to_drive()                                  # whatever finished since the last minute
            print(f'\n{n_done()}/{N_TOTAL} done ({(time.time() - t0) / 60:.0f} min this session)')
            if fatal:
                raise RuntimeError(fatal)
            r = subprocess.run([sys.executable, f'{REPO}/evaluate.py', OUT], capture_output=True, text=True, cwd=REPO)
            print(r.stdout[-6000:], r.stderr[-2000:])
            recs = [json.loads(l) for l in open(f'{OUT}/eval.jsonl')]
            stats = {}
            for f in glob.glob(f'{OUT}/mbpp_*.json'):
                d = json.load(open(f)); stats[d['name']] = d
            import statistics
            walls = [d['wall_s'] for d in stats.values()]
            print(f'time: total {sum(walls) / 3600:.2f} h, median {statistics.median(walls):.1f} s per task')
            if MODE == 'itergen_on':
                v = [x for d in stats.values() for x in d['stats'].get('violations', [])]
                print(f"checks fired {len(v)} times on {sum(1 for d in stats.values() if d['stats'].get('violations'))} tasks:",
                      dict(__import__('collections').Counter(x['rule'] for x in v)))
            if MODE.startswith('itergen'):
                print('stop reasons:', dict(__import__('collections').Counter(d['stats'].get('stop_reason') for d in stats.values())),
                      '| fallback used:', sum(d['stats'].get('fallback_used', False) for d in stats.values()))
            if MODE == 'chopchop':
                s = [d['stats'] for d in stats.values()]
                print('checks', sum(x['checks'] for x in s), '| rejected', sum(x['rejected'] for x in s),
                      '| CPU limit hit', sum(x.get('limit_hit', False) for x in s),
                      '| wall cap hit', sum(x.get('wall_cap_hit', False) for x in s),
                      '| check errors', sum(x.get('check_errors', 0) for x in s))
                # results under every lower limit, rebuilt from the per-check trace
                r = subprocess.run([sys.executable, f'{REPO}/tools/limit_curve.py', OUT], capture_output=True, text=True, cwd=REPO)
                print(r.stdout[-4000:], r.stderr[-2000:])
            copy_to_drive()
            ATTRIBUTE = True
            if MODE == 'chopchop' and ATTRIBUTE:
                p = subprocess.Popen([sys.executable, '-u', f'{REPO}/tools/attribute_rejections.py', OUT],
                                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO, env=ENV)
                for line in p.stdout:
                    print(line.rstrip(), flush=True)
                p.wait()
                copy_to_drive()
        except Exception:
            traceback.print_exc()
            print('!!! this run did not finish; going on with the next one (Run all again later)', flush=True)
            continue
        if n_done() < N_TOTAL:
            print(f'!!! only {n_done()}/{N_TOTAL} tasks done; going on with the next run (Run all again later)', flush=True)
            continue
        if DRIVE_OK:
            open(f'{DRIVE_OUT}/queue_done.txt', 'w').write(time.strftime('%Y-%m-%d %H:%M:%S\n'))
    print('queue finished')
except KeyboardInterrupt:
    interrupted = True
    raise
finally:
    if AUTO_DISCONNECT and not interrupted:
        print(time.strftime('%H:%M:%S'), 'AUTO_DISCONNECT: writing to Drive, then releasing the runtime', flush=True)
        try:
            from google.colab import drive, runtime
            if DRIVE_OK:
                drive.flush_and_unmount()
            time.sleep(30)
            runtime.unassign()
        except Exception as e:
            print('could not release the runtime:', type(e).__name__, e)
